In [ ]:
# passt pca summary stats
import pandas as pd
import numpy as np
import glob
import os
from scipy import stats
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

PCA_DIR = '/home/mellie/supervised-vs-SSL/PCA_results/passt'
OUT_SUMMARY = os.path.join(PCA_DIR, '0_layer_summary_stats.csv')

manual_layers = [
    'net_blocks_0_pca.csv', 'net_blocks_1_pca.csv', 'net_blocks_2_pca.csv', 'net_blocks_3_pca.csv',
    'net_blocks_4_pca.csv', 'net_blocks_5_pca.csv', 'net_blocks_6_pca.csv', 'net_blocks_7_pca.csv',
    'net_blocks_8_pca.csv', 'net_blocks_9_pca.csv', 'net_blocks_10_pca.csv', 'net_blocks_11_pca.csv',
    'z_final_embeddings_pca.csv'
]
csv_paths = [os.path.join(PCA_DIR, f'{name}') for name in manual_layers]
rows = []
for path in csv_paths:
    layer = os.path.basename(path).replace('_pca.csv','')
    df = pd.read_csv(path)
    # expect df contains columns: feature_file, task, pc1, pc2
    if not {'pc1','pc2','task'}.issubset(df.columns):
        print('Skipping', path, '- missing expected columns')
        continue
    X = df[['pc1','pc2']].values
    task = df['task'].astype(int).values

    # silhouette (requires >1 cluster in input)
    try:
        sil = silhouette_score(X, task)
    except Exception:
        sil = np.nan

    # ANOVA on PC1 and PC2
    groups_pc1 = [df.loc[task==g, 'pc1'].values for g in np.unique(task)]
    groups_pc2 = [df.loc[task==g, 'pc2'].values for g in np.unique(task)]
    anova_pc1 = stats.f_oneway(*groups_pc1) if len(groups_pc1) > 1 else (np.nan, np.nan)
    anova_pc2 = stats.f_oneway(*groups_pc2) if len(groups_pc2) > 1 else (np.nan, np.nan)

    # centroids & normalized distance
    centroids = df.groupby('task')[['pc1','pc2']].mean()
    if centroids.shape[0] == 2:
        d = np.linalg.norm((centroids.loc[1]-centroids.loc[0]).values)
        # pooled std
        s0 = df.loc[task==0, ['pc1','pc2']].values
        s1 = df.loc[task==1, ['pc1','pc2']].values
        pooled_std = np.sqrt(((s0.var(axis=0).mean() + s1.var(axis=0).mean())/2))
        norm_d = d / (pooled_std + 1e-8)
    else:
        d, norm_d = np.nan, np.nan

    rows.append({
        'layer': layer,
        'n_samples': len(df),
        'silhouette': sil,
        'anova_pc1_F': float(anova_pc1.statistic) if not np.isnan(anova_pc1.statistic) else np.nan,
        'anova_pc1_p': float(anova_pc1.pvalue) if not np.isnan(anova_pc1.pvalue) else np.nan,
        'anova_pc2_F': float(anova_pc2.statistic) if not np.isnan(anova_pc2.statistic) else np.nan,
        'anova_pc2_p': float(anova_pc2.pvalue) if not np.isnan(anova_pc2.pvalue) else np.nan,
        'centroid_dist': d,
        'centroid_dist_norm': norm_d,
        'csv': path,
    })
csv_paths = [os.path.join(PCA_DIR, name) for name in manual_layers]

summary = pd.DataFrame(rows).sort_values('layer')
summary.to_csv(OUT_SUMMARY, index=False)
print('Wrote', OUT_SUMMARY)

# quick plot: normalized centroid distance and silhouette across layers
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.plot(summary['layer'], summary['centroid_dist_norm'], marker='o')
plt.xticks(rotation=90); plt.title('Centroid distance (norm)')
plt.subplot(1,2,2)
plt.plot(summary['layer'], summary['silhouette'], marker='o', color='C1')
plt.xticks(rotation=90); plt.title('Silhouette score')
plt.tight_layout()
plt.savefig(os.path.join(PCA_DIR, 'diagnostic_summary.png'), dpi=200)
plt.close()
print('Diagnostic plot saved.')

Wrote /home/mellie/supervised-vs-SSL/PCA_results/passt/0_layer_summary_stats.csv
Diagnostic plot saved.


In [ ]:
# byola pca summary stats
import pandas as pd
import numpy as np
import glob, os
from scipy import stats
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

PCA_DIR = '/home/mellie/supervised-vs-SSL/PCA_results/byola'
OUT_SUMMARY = os.path.join(PCA_DIR, '0_layer_summary_stats.csv')

csv_paths = sorted(glob.glob(os.path.join(PCA_DIR, '*.csv')))
rows = []
for path in csv_paths:
    layer = os.path.basename(path).replace('.csv','')
    df = pd.read_csv(path)

    # accept different column names: prefer 'task' then 'label'
    if 'task' in df.columns:
        task_col = df['task']
    elif 'label' in df.columns:
        task_col = df['label']
    else:
        print('Skipping', path, '- no 'task' or 'label' column')
        continue

    # map to binary 0=music, 1=speech
    if pd.api.types.is_numeric_dtype(task_col):
        task = task_col.astype(int).values
    else:
        def map_to_binary(s):
            s = str(s).lower()
            if 'speech' in s: return 1
            if 'music' in s: return 0
            # fallback: if token looks like a speech command (short alpha tokens), treat as speech
            if len(s) <= 6 and s.isalpha(): return 1
            return np.nan
        mapped = task_col.astype(str).apply(map_to_binary)
        if mapped.isna().any():
            print(f'Skipping {path} - unmapped labels present (sample: {mapped[mapped.isna()].head(3).tolist()})')
            continue
        task = mapped.astype(int).values

    # expect df contains pc1, pc2
    if not {'pc1','pc2'}.issubset(df.columns):
        print('Skipping', path, '- missing pc1/pc2')
        continue

    X = df[['pc1','pc2']].values

    # silhouette
    try:
        sil = silhouette_score(X, task)
    except Exception:
        sil = np.nan

    # ANOVA on PC1 and PC2
    groups_pc1 = [df.loc[task==g, 'pc1'].values for g in np.unique(task)]
    groups_pc2 = [df.loc[task==g, 'pc2'].values for g in np.unique(task)]
    anova_pc1 = stats.f_oneway(*groups_pc1) if len(groups_pc1) > 1 else (np.nan, np.nan)
    anova_pc2 = stats.f_oneway(*groups_pc2) if len(groups_pc2) > 1 else (np.nan, np.nan)

    # centroids & normalized distance
    centroids = df.groupby(pd.Series(task, name='task'))[['pc1','pc2']].mean()
    if centroids.shape[0] == 2:
        d = np.linalg.norm((centroids.loc[1]-centroids.loc[0]).values)
        s0 = df.loc[task==0, ['pc1','pc2']].values
        s1 = df.loc[task==1, ['pc1','pc2']].values
        pooled_std = np.sqrt(((s0.var(axis=0).mean() + s1.var(axis=0).mean())/2))
        norm_d = d / (pooled_std + 1e-8)
    else:
        d, norm_d = np.nan, np.nan

    rows.append({
        'layer': layer,
        'n_samples': len(df),
        'silhouette': sil,
        'anova_pc1_F': float(anova_pc1.statistic) if hasattr(anova_pc1, 'statistic') else np.nan,
        'anova_pc1_p': float(anova_pc1.pvalue) if hasattr(anova_pc1, 'pvalue') else np.nan,
        'anova_pc2_F': float(anova_pc2.statistic) if hasattr(anova_pc2, 'statistic') else np.nan,
        'anova_pc2_p': float(anova_pc2.pvalue) if hasattr(anova_pc2, 'pvalue') else np.nan,
        'centroid_dist': d,
        'centroid_dist_norm': norm_d,
        'csv': path,
    })

summary = pd.DataFrame(rows).sort_values('layer')
summary.to_csv(OUT_SUMMARY, index=False)
print('Wrote', OUT_SUMMARY)

# quick plot: normalized centroid distance and silhouette across layers
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.plot(summary['layer'], summary['centroid_dist_norm'], marker='o')
plt.xticks(rotation=90); plt.title('Centroid distance (norm)')
plt.subplot(1,2,2)
plt.plot(summary['layer'], summary['silhouette'], marker='o', color='C1')
plt.xticks(rotation=90); plt.title('Silhouette score')
plt.tight_layout()
plt.savefig(os.path.join(PCA_DIR, 'diagnostic_summary.png'), dpi=200)
plt.close()
print('Diagnostic plot saved.')

In [ ]:
# compute binary task accuracy instead of multi-class predictions
import pandas as pd, numpy as np
def is_speech(s):
    s=str(s).lower()
    speech_tokens={'right','no','off','up','yes','down','stop','go','marvin','sheila'}
    if s in speech_tokens: return 1
    if 'speech' in s: return 1
    # otherwise assume music for your domain; adapt if needed
    return 0
df=pd.read_csv('supervised_vs_ssl/passt_predictions.csv')
df['actual_task']=df['actual_label'].apply(is_speech)
df['predicted_task']=df['predicted_label'].apply(is_speech)
ok=df[['actual_task','predicted_task']].notna().all(axis=1)
print('binary accuracy:', (df.loc[ok,'actual_task']==df.loc[ok,'predicted_task']).mean())

In [ ]:
# replace the problematic shuffle-check cell with this robust version

import pandas as pd, numpy as np
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

df = pd.read_csv('/home/mellie/supervised-vs-SSL/PCA_results/byola/pca_fc_0.csv')

# find label column and map to binary
if 'label' in df.columns:
    raw = df['label'].astype(str).str.strip().str.lower()
elif 'task' in df.columns:
    raw = df['task'].astype(str).str.strip().str.lower()
else:
    # try common alternates
    for c in ['actual_label','y','class','label_str']:
        if c in df.columns:
            raw = df[c].astype(str).str.strip().str.lower()
            break
    else:
        raise ValueError('No label column found in dataframe (checked 'label','task',etc)')

# simple explicit mapping
map_dict = {'music': 0, 'speech': 1}
mapped = raw.map(map_dict)

# diagnostics
print('unique raw label examples:', raw.unique()[:10])
print('mapped value counts (including NaN):\n', mapped.value_counts(dropna=False))

# select only rows with mapped labels and required pcs
keep_idx = mapped.notna() & df[['pc1','pc2']].notnull().all(axis=1)
n_keep = keep_idx.sum()
if n_keep == 0:
    raise ValueError('No rows with mapped labels and pc1/pc2 present. Inspect raw labels above.')

X = df.loc[keep_idx, ['pc1','pc2']].values
y = mapped[keep_idx].astype(int).values

print('Using', X.shape[0], 'samples: class counts ->', np.bincount(y))

# ensure at least two classes present
if len(np.unique(y)) < 2:
    raise ValueError('Only one class present after mapping; cannot compute silhouette.')

orig_sil = silhouette_score(X, y)
y_shuf = np.random.permutation(y)
shuf_sil = silhouette_score(X, y_shuf)
print('orig sil', orig_sil, 'shuffled sil', shuf_sil)

clf = LogisticRegression(max_iter=500)
print('cv acc', cross_val_score(clf, X, y, cv=5).mean())

df = pd.read_csv('/home/mellie/supervised-vs-SSL/PCA_results/byola/pca_fc_0.csv')
print('does the filename correlate with task:', df['feature_file'].str.contains('|'.join(['right','yes','no','up','down','go'])) .mean())

unique raw label examples: ['music' 'speech']
mapped value counts (including NaN):
 task
0    320
1    180
Name: count, dtype: int64
Using 500 samples: class counts -> [320 180]
orig sil 0.13095779947145228 shuffled sil 0.0007719467602121939
cv acc 0.6880000000000001
does the filename correlate with task: 0.554


In [ ]:
# ANOVA on statistically significant differences between classes across layers
from scipy import stats
import pandas as pd

def layer_anova_analysis(features_dict, labels, task_labels):
    """
    Perform ANOVA analysis on each layer's features.
    
    Args:
        features_dict: Dictionary with layer names as keys and PCA-transformed features as values
        labels: Original class labels (music genre or speech type)
        task_labels: Binary labels (0=music, 1=speech)
    
    Returns:
        DataFrame with ANOVA results for each layer
    """
    results = []
    
    for layer_name, features_pca in features_dict.items():
        # for overall task discrimination (music vs speech)
        f_val_task, p_val_task = stats.f_oneway(
            features_pca[task_labels == 0, :2].T,  # music samples, first 2 PCs
            features_pca[task_labels == 1, :2].T   # speech samples, first 2 PCs
        )
        
        # separation within music classes
        music_samples = features_pca[task_labels == 0]
        music_labels = labels[task_labels == 0]
        music_groups = [music_samples[music_labels == label, :2].T 
                       for label in np.unique(music_labels)]
        f_val_music, p_val_music = stats.f_oneway(*music_groups)
        
        # separation within speech classes
        speech_samples = features_pca[task_labels == 1]
        speech_labels = labels[task_labels == 1]
        speech_groups = [speech_samples[speech_labels == label, :2].T 
                        for label in np.unique(speech_labels)]
        f_val_speech, p_val_speech = stats.f_oneway(*speech_groups)
        
        results.append({
            'layer': layer_name,
            'task_f_value': f_val_task,
            'task_p_value': p_val_task,
            'music_f_value': f_val_music,
            'music_p_value': p_val_music,
            'speech_f_value': f_val_speech, 
            'speech_p_value': p_val_speech
        })
    
    return pd.DataFrame(results)

In [ ]:
# add silhouette scores
from sklearn.metrics import silhouette_score

def compute_silhouette_scores(features_dict, task_labels):
    """
    Computes silhouette scores for task separation at each layer.
    """
    results = []
    
    for layer_name, features in features_dict.items():
        # standardize and apply PCA
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(features)
        pca = PCA(n_components=2)
        features_pca = pca.fit_transform(features_scaled)
        
        # compute silhouette score (how well separated the clusters are)
        sil_score = silhouette_score(features_pca, task_labels)
        
        results.append({
            'layer': layer_name,
            'silhouette_score': sil_score
        })
    
    return pd.DataFrame(results)